<!-- gen:values.yml#weeks.02b.notebook-title|Introduction -->
# Week 2b Introduction: Advanced Python
<!-- /gen -->

*Three strategies for reading code you didn't write, and the four Python patterns your [Associate](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-associate) uses most.*

## Overview

From this week on, most of the code in front of you is code you did not write. Your Associate drafts it in seconds. You are the [analyst](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-analyst) who reads the draft and takes responsibility for the memo.

Two things happen at once:

1. **You learn to read code you didn't write**: Three strategies, practised on realistic Associate drafts
2. **You learn the vocabulary those drafts use**: Comprehensions, `lambda` with `sorted`, `try`/`except`, and a first small class

You work in your **GitHub Codespace**, the workstation from the Week 2 workshop. GitHub Copilot is your Associate, and [SpecStory](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-specstory) builds your [AI Use Record](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-ai-use-record). The [Your workstation guide](../../Guides/codespaces-start.md) covers the workstation.

## This week's learning objectives

The [week README](./README.md) lists this week's learning objectives. [Quest 02b](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-quest) asks you to do all of them for a client, except reading a class. The Demonstration builds that skill, and the quest's [Gold](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-gold) tier and the closing quiz use it.

## How to read code you didn't write

An Associate's first draft usually looks right. It is not always right. Reading it well is a procedure. This module uses three strategies, always in this order:

1. **Trace the data flow**: What goes in, what comes out, and what shape is it at each step?
2. **Name the <!-- plain-language: everyday intent -->intent of each block**: What is each block for, in one plain sentence?
3. **Run fragments**: What does this piece do on its own, on real data?

Practise on a real draft. First, run the cell below. It builds one Saturday of sales at Beans of Egham, the Week 1 coffee cart, as a list of dictionaries.

In [ ]:
# --- Data setup: Run this cell first (runs anywhere, no download needed) ---
import numpy as np

rng = np.random.default_rng(5813)  # module seed: Do not change

# Beans of Egham (the Week 1 coffee cart) had a busy Saturday.
# One dict per transaction: What was sold, how many cups, price per cup.
MENU = {"flat white": 3.40, "latte": 3.60, "espresso": 2.20,
        "mocha": 3.80, "breakfast tea": 2.40}
drinks = list(MENU)

saturday_sales = []
for n in range(40):
    drink = str(rng.choice(drinks, p=[0.3, 0.25, 0.15, 0.15, 0.15]))
    saturday_sales.append({
        "order": n + 1,
        "drink": drink,
        "cups": int(rng.integers(1, 4)),
        "price_per_cup": MENU[drink],
    })

print(f"Created saturday_sales: {len(saturday_sales)} transactions")
print(f"First transaction: {saturday_sales[0]}")

### Strategy 1: Trace the data flow

You ask your Associate *"which drink made the most money on Saturday?"* and it hands you the cell below. Do not read it line by line yet. First trace the data. What does each named thing hold, and what shape is it?

- `saturday_sales`: A **list of dictionaries**, one per transaction (the data setup cell makes it)
- `revenue_by_drink`: A **dictionary** mapping drink name to total pounds
- `top`: A **list of tuples** (pairs of drink and total), sorted, biggest first

The data goes in as a list of transactions, becomes totals per drink, and comes out as a ranked list. Before you run a line, you know what a sensible answer looks like. You expect five drinks, positive totals, and totals that add up to the day's takings. Code that breaks that expectation is code to question.

Run the draft.

In [ ]:
# The Associate's draft: "which drink made the most money?"
revenue_by_drink = {}
for sale in saturday_sales:
    drink = sale["drink"]
    takings = sale["cups"] * sale["price_per_cup"]
    revenue_by_drink[drink] = revenue_by_drink.get(drink, 0) + takings

top = sorted(revenue_by_drink.items(), key=lambda pair: pair[1], reverse=True)

for drink, total in top:
    print(f"{drink:15s} £{total:.2f}")

### Strategy 2: Name the intent of each block

Read the draft again. Give each block a one-sentence job description in your own words. Write it as a comment or say it out loud:

1. *Lines 2-6: Total up the takings for each drink, one running total per drink*
2. *Line 8: Rank the drinks, highest takings first*
3. *Lines 10-11: Print the ranking as pounds and pence*

If you cannot name what a block is for, that block is where the bug lives. "Some sorting thing with a lambda" is not a name. It is a gap in your understanding, and wrong numbers hide there.

### Strategy 3: Run fragments

Pull out anything you cannot name. Run it on its own, on one piece of data, in a spare cell. Two examples follow. The first shows what `.items()` produces.

In [ ]:
# Fragment 1: What does .items() actually produce?
list(revenue_by_drink.items())[:2]

The second fragment runs the `lambda` on its own, fed a single pair by hand.

In [ ]:
# Fragment 2: What does the lambda do to ONE of those pairs?
grab_second = lambda pair: pair[1]
grab_second(("latte", 87.60))

So `key=lambda pair: pair[1]` means: *When ranking, look at the money, not the name.*

The next cell is an [AI Lens](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-ai-lens).

> 🤖 **AI Lens:** The three strategies work on your Associate too. Paste a draft in place of the placeholder and ask the question below. Then check its answer against fragments you run yourself.

```text
<paste the draft here>

Walk me through the data flow: what is the type and shape of each variable?
```

**A good answer** names every variable in the draft with its type and shape. For example, the draft above has a list of dicts, a dict of names to totals, and a list of pairs. Each claim in the answer survives a fragment you run yourself.

## This week's vocabulary

The draft above used a plain loop, a `lambda`, and `sorted(key=...)`. Your Associate's drafts use four patterns again and again. The [Demonstration](./Demonstration.ipynb) does each one properly.

### 1. Comprehensions: A list built in one move

A comprehension is Week 1's loop with `.append()`, folded into one line. The official tour is [list comprehensions in the Python tutorial](https://docs.python.org/3/tutorial/datastructures.html#list-comprehensions). The cell below builds the same list both ways.

In [ ]:
# Week 1 style:
large_orders = []
for sale in saturday_sales:
    if sale["cups"] >= 3:
        large_orders.append(sale["order"])

# This week (same result, one move):
large_orders = [sale["order"] for sale in saturday_sales if sale["cups"] >= 3]

print(f"Orders of 3+ cups: {large_orders}")

### 2. `lambda` + `sorted(key=...)`: Rank anything by anything

`sorted(things, key=...)` ranks a collection. The `key` function says what to rank by. Read `lambda sale: sale["cups"]` as "given a sale, use its cups". The [Sorting HOW TO](https://docs.python.org/3/howto/sorting.html) covers the whole pattern. The cell below ranks Saturday's orders by cups.

In [ ]:
biggest = sorted(saturday_sales, key=lambda sale: sale["cups"], reverse=True)
print(f"Biggest single order: {biggest[0]}")

### 3. `try`/`except`: Failing on purpose, loudly

Real data has missing fields and text in number columns. `try`/`except` lets code survive a failure you expect without hiding it. The Python tutorial covers it in [errors and exceptions](https://docs.python.org/3/tutorial/errors.html). A large part of Quest 02b is the difference between handling an error and silencing one. The cell below tries to price an order with no price recorded.

In [ ]:
sale = {"drink": "latte", "cups": 2}   # no price recorded!

try:
    takings = sale["cups"] * sale["price_per_cup"]
except KeyError as missing:
    print(f"⚠️ Order can't be priced (missing field: {missing})")
    takings = None

### 4. A small class: Data and behaviour, boxed together

When a dictionary and its helper functions always travel together, you can box them into a **class**. The [tutorial chapter on classes](https://docs.python.org/3/tutorial/classes.html) goes deeper. In Week 4 you meet pandas, which is built from classes. This week's goal is reading a class, not writing one. The cell below defines a loyalty card: Its data lives in `__init__`, and its methods are verbs.

In [ ]:
class LoyaltyCard:
    """One customer's coffee stamps at Beans of Egham."""

    def __init__(self, owner):
        self.owner = owner      # the data lives in __init__ ...
        self.stamps = 0

    def add_stamps(self, cups):   # ... and the methods are verbs
        self.stamps = self.stamps + cups

card = LoyaltyCard("Priya")
card.add_stamps(3)
print(f"{card.owner} has {card.stamps} stamps")

## The week ahead

The week ends with **Quest 02b: The Subscription Churn Memo**. StreamShore, a streaming startup, wants churn numbers the board can trust.

The [week README](./README.md) lists what to work through at home, in your own time.